# E-Means: Methodological Foundations & Paper Analysis### Reference*Bessa et al. (Meta FAIR & ENPC, 2024/2026), "e-means: outperforming k-means (in one-tenth the epochs)", arXiv:2411.09501 [cs.LG].*---## 1. Problem FormulationGiven an observation matrix $X = \{x_1, \dots, x_n\} \subset \mathbb{R}^d$ and a target number of clusters $k$, $k$-means seeks a partition $C = \{C_1, \dots, C_k\}$ and corresponding centroids $\{c_1, \dots, c_k\}$ that minimize the Within-Cluster Sum of Squares (Inertia):$$\mathcal{J}(C) = \sum_{j=1}^k \sum_{x_i \in C_j} \|x_i - c_j\|_2^2$$### The Lloyd BottleneckLloyd's algorithm alternates between exact assignment and full-batch recomputation:1. $y_i = \arg\min_j \|x_i - c_j\|_2^2$2. $c_j = \frac{1}{|C_j|} \sum_{x \in C_j} x$Each iteration requires $O(n \cdot k \cdot d)$ distance computations. On massive embedding collections, computing dozens of full sweeps becomes computationally prohibitive.### Classical Mini-Batch Limitations- **MacQueen Accumulation:** Centroids update as cumulative running averages. Early misallocations remain permanently embedded in centroid statistics.- **Gradient Descent:** Sensitive to step size selection; centroids can oscillate or leave the convex hull of the data.- **Cluster Starvation:** In mini-batch settings, centroids trapped in sparse regions lose all assignments ("die"), wasting codebook capacity (often 20% to 30% of centroids).---## 2. The E-Means ArchitectureE-Means resolves these issues by reformulating mini-batch centroid tracking via an Exponential Moving Average (EMA) over sufficient statistics:1. **Batch Sufficient Statistics:** For mini-batch $B_t \subset X$:   $$n_j = \sum_{i \in B_t} \mathbb{I}(y_i = j), \quad s_j = \sum_{i \in B_t, y_i = j} x_i$$2. **EMA Updates:**   $$N_j \leftarrow (1 - \alpha_t) N_j + \alpha_t n_j$$   $$S_j \leftarrow (1 - \alpha_t) S_j + \alpha_t s_j$$   $$c_j = \frac{S_j}{N_j} \quad (\text{when } N_j > 0)$$3. **Half-Cosine Annealing Schedule:**   $$\alpha(t) = \alpha_0 \left[ 0.1 + 0.9 \cdot \frac{1}{2} \left( 1 + \cos\left( \frac{\pi t}{M} \right) \right) \right]$$   where $M = T \cdot \lceil n / B \rceil$. This provides wide spatial exploration during early iterations and smooth refinement in late stages.4. **Dead-Cluster Revival via Proportional Donor Splitting:**   If running mass $N_j < 0.1 \cdot \bar{N}$, a donor cluster $d$ is sampled proportional to $N_d$ and split with directional perturbation $\delta \sim \mathcal{N}(0, \sigma^2 I)$.5. **Exact 1-Pass Finalization:**   A single full-dataset Lloyd update clears remaining mini-batch variance, restoring global centroid optimality for the final assignments.

In [ ]:
import sys
sys.path.insert(0, "..")
import math
import numpy as np
import matplotlib.pyplot as plt

# Half-cosine annealing schedule across total update steps
total_steps = 100
alpha_0 = 1.0

steps = np.arange(total_steps)
alphas = [alpha_0 * (0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * t / (total_steps - 1)))) for t in steps]

plt.figure(figsize=(7.5, 3.8), dpi=150)
plt.plot(steps, alphas, color="#1f77b4", linewidth=2.2, label="E-Means Schedule alpha(t)")
plt.axhline(0.1, color="#d62728", linestyle="--", alpha=0.7, label="Floor epsilon = 0.1 * alpha_0")
plt.title("Half-Cosine Step Decay Schedule", fontsize=11, fontweight="bold")
plt.xlabel("Mini-Batch Step (t)", fontsize=9)
plt.ylabel("Effective Update Rate", fontsize=9)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

## Theoretical Properties- **Convex Hull Invariance:** Because $c_j = S_j / N_j$ is a convex combination of data points, centroids are strictly bounded within the data manifold.- **Smooth Forgetfulness:** Decaying $N_j$ and $S_j$ exponentially discards noisy initialization steps without manual warm-up phases.- **Codebook Health:** Proportional splitting reduces inactive codebook vectors to $< 0.3\%$, compared to $> 25\%$ in unregularized mini-batch k-means.